In [2]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import duckdb




In [3]:

##Phase 1: Data inspection... We want to know the data type, the number of missing values in the dataset, 
##and to have an idea of the dataset by printing the first 5 rows.
df=pd.read_csv(r"C:\Users\Usuari\Downloads\reporte_cruceros_revenue_management.csv")
print(df.head())
df.info()
df.describe()
df.isna().sum()





      RES_ID Fecha_Viaje Fecha_Reserva  Lead_Time_Dias  Noches_Estancia  \
0  RES-00001  2018-01-05    2017-08-18             140                6   
1  RES-00002  2018-01-05    2017-09-14             113                6   
2  RES-00003  2018-01-05    2017-11-01              65                6   
3  RES-00004  2018-01-05    2017-10-31              66                6   
4  RES-00005  2018-01-05    2017-09-17             110                6   

              Barco      Compañia     Tipo_Ruta Suite_Type Booking_Source  \
0  MSC World Europa  MSC Cruceros  Mediterráneo    Balcony         Direct   
1  MSC World Europa  MSC Cruceros  Mediterráneo   Interior            B2B   
2  MSC World Europa  MSC Cruceros  Mediterráneo  Oceanview            Web   
3  MSC World Europa  MSC Cruceros  Mediterráneo   Interior            Web   
4  MSC World Europa  MSC Cruceros  Mediterráneo  Oceanview         Direct   

               Package Guest_Country  Cabinas_Totales_Barco  \
0        All Inclusive 

RES_ID                               0
Fecha_Viaje                          0
Fecha_Reserva                        0
Lead_Time_Dias                       0
Noches_Estancia                      0
Barco                                0
Compañia                             0
Tipo_Ruta                            0
Suite_Type                           0
Booking_Source                       0
Package                              0
Guest_Country                        0
Cabinas_Totales_Barco                0
Cabinas_Reservadas                   0
Porcentaje_Ocupacion_Ciclo           0
Pasajeros_Reserva                    0
Tripulacion                          0
Ingreso_Total_Reserva_USD            0
Gasto_Promedio_Diario_Huesped_USD    0
Puntuacion_Satisfaccion              0
dtype: int64


We describe the dataset; It contains 20 columns with different information
#0   RES_ID                             77040 non-null  object 

#1   Fecha_Viaje                        77040 non-null  object 

 #2   Fecha_Reserva                      77040 non-null  object 
 
 #3   Lead_Time_Dias                     77040 non-null  int64  
 
 #4   Noches_Estancia                    77040 non-null  int64  
 
 #5   Barco                              77040 non-null  object 
 
 #6   Compañia                           77040 non-null  object 
 
 #7   Tipo_Ruta                          77040 non-null  object 
 
 #8   Suite_Type                         77040 non-null  object 
 
 #9   Booking_Source                     77040 non-null  object 
 
 #10  Package                            77040 non-null  object 
 
 #11  Guest_Country                      77040 non-null  object 
 
 #12  Cabinas_Totales_Barco              77040 non-null  int64
 
 #13  Cabinas_Reservadas                 77040 non-null  int64  
 
 #14  Porcentaje_Ocupacion_Ciclo         77040 non-null  float64
 
 #15  Pasajeros_Reserva                  77040 non-null  int64  
 
 #16  Tripulacion                        77040 non-null  int64  
 
 #17  Ingreso_Total_Reserva_USD          77040 non-null  float64
 
 #18  Gasto_Promedio_Diario_Huesped_USD  77040 non-null  float64
 
 #19  Puntuacion_Satisfaccion            77040 non-null  float64
#dtypes: float64(4), int64(6), object(10)

We proceed with data cleaning; one must drop the duplicate rows, check that every column has the correct datatype; finally one msut prove that all the numerical data is logical and is included between the established ranges

In [8]:
#Data cleaning. Format all the columns to the correct data type and delete duplicate rows 
df.drop_duplicates(inplace=True)
df['Fecha_Viaje']=pd.to_datetime(df['Fecha_Viaje'])
df['Fecha_Reserva']=pd.to_datetime(df['Fecha_Reserva'])
# CORRECTED VERSION
df = df[
    (df['Lead_Time_Dias'] > 0)
    & (df['Cabinas_Totales_Barco'] > 0)
    & (df['Cabinas_Reservadas'] > 0)
    & (df['Porcentaje_Ocupacion_Ciclo'] < 100)
    & (df['Pasajeros_Reserva'] > 0)  
    & (df['Tripulacion'] > 0)
    & (df['Puntuacion_Satisfaccion'] < 5.0)  
]
#We remove wrong data; check that all the columns have coherent data





Now comes the most important part of the job: connecting SQL to Python. We'll extract views from the raw data; these views are created to answer the following questions:

Which ship / company has obtained the best results? (Query 1)

Which kind of route is the most popular and profitable? (Query 2)

We focus on one of the most important aspects of cruise ship reservations: lead time. We compare average satisfaction, spending by customer, and generated income across different levels of journey planning. Query 3

Finally, Query 4 addresses a critical facet of cruise ship reservation management: seasonality. We conduct a comparative analysis of aggregate income generated across the different seasons of the year.




In [9]:
#WE extract insights from data connecting SQL by Duck Library

#We extract general results from database: total revenues, the average spending by customer, and the average revenue
query_1_results= """ 
SELECT Barco,
COUNT() AS total_reservas,
SUM(Ingreso_Total_Reserva_USD) AS total_ingreso,
RANK() OVER(ORDER BY SUM(Ingreso_Total_Reserva_USD) DESC) AS ranking_revenues,
AVG(Ingreso_Total_Reserva_USD) AS ingreso_reserva,
AVG(Gasto_Promedio_Diario_Huesped_USD) AS gasto_persona,
AVG(Puntuacion_Satisfaccion) AS rate,
RANK() OVER(ORDER BY AVG(Puntuacion_Satisfaccion) ) AS ranking_rate,
FROM df 
GROUP BY Barco 
ORDER BY AVG(Puntuacion_Satisfaccion)

"""


query_2_route= """
SELECT 
Barco AS Ship,
Tipo_Ruta AS Route,
COUNT(*) AS Total_reservations,
AVG(Ingreso_Total_Reserva_USD) AS avg_revenue_bycustomer,
DENSE_RANK() OVER (ORDER BY AVG(Ingreso_Total_Reserva_USD) DESC) AS ranking_revenue
FROM df
GROUP BY Barco, Tipo_Ruta
"""


query_3_reservations="""
SELECT 
CASE
            WHEN Lead_Time_Dias<=30 THEN  'Last-hour'
            WHEN Lead_Time_Dias BETWEEN 31 AND 90 THEN 'Short-term'
            WHEN Lead_Time_Dias BETWEEN 90 AND 180 THEN 'Planned'
            WHEN Lead_Time_Dias> 180 THEN 'High-Anticipation' 
            END AS Lead_Time_Days,
COUNT(*) AS reservations_antelation,
AVG(Gasto_Promedio_Diario_Huesped_USD) AS avg_customerspend,
DENSE_RANK() OVER (ORDER BY AVG(Gasto_Promedio_Diario_Huesped_USD )DESC) AS ranking_customerspend, 
AVG( Ingreso_Total_Reserva_USD ) AS avg_customer_spend,
DENSE_RANK() OVER (ORDER BY AVG(Ingreso_Total_Reserva_USD)DESC) AS ranking_customerrevenue,
AVG(Puntuacion_Satisfaccion) AS avg_rating,
DENSE_RANK() OVER (ORDER BY AVG(Puntuacion_Satisfaccion) DESC) AS ranking_satisfaction
FROM df 
GROUP BY Lead_Time_Days
ORDER BY CASE  Lead_Time_Days
            WHEN  'Last-hour' THEN 1  
            WHEN  'Short-term' THEN 2
            WHEN  'Planned' THEN 3
            WHEN  'High-Anticipation'THEN 3 
END;

 
"""
query_4_seasonality="""
SELECT 
    CASE 
        WHEN EXTRACT (MONTH FROM Fecha_Viaje) BETWEEN 1 AND 3 THEN 'Winter'
        WHEN  EXTRACT (MONTH FROM Fecha_Viaje) BETWEEN 4 AND 9 THEN 'Summer High-Season'
        WHEN EXTRACT (MONTH FROM Fecha_Viaje) BETWEEN 9 AND 12 THEN  'Fall-Christmas'
END AS Season,
COUNT(*) AS  number_reservations,
SUM(Ingreso_Total_Reserva_USD) AS total_income,
RANK() OVER(PARTITION BY Season ORDER BY SUM(Ingreso_Total_Reserva_USD) DESC) AS ranking_total_revenue,
AVG( Ingreso_Total_Reserva_USD ) AS avg_customer_spend,
DENSE_RANK() OVER (PARTITION BY Season ORDER BY AVG(Ingreso_Total_Reserva_USD) DESC) AS ranking_customerrevenue,
AVG(Puntuacion_Satisfaccion) AS rate,
RANK() OVER(PARTITION BY Season ORDER BY AVG(Puntuacion_Satisfaccion) DESC) AS ranking_rate
FROM df
GROUP BY Season
ORDER BY CASE  Season
            WHEN  'Winter' THEN 1  
            WHEN  'Summer High-season' THEN 2
            WHEN  'Fall-Christmas' THEN 3
END;

"""
df_view1=duckdb.query(query_1_results).df()
df_view2=duckdb.query(query_2_route).df()
df_view3=duckdb.query(query_3_reservations).df()
df_view4=duckdb.query(query_4_seasonality).df()

df_view1.to_csv('vista_barcos.csv', index=False)
df_view2.to_csv('vista_rutas.csv', index=False)
df_view3.to_csv('vista_lead_time.csv', index=False)
df_view4.to_csv('vista_season_.csv', index=False)